---

In [0]:
display(spark.sql("SELECT COUNT(*) FROM bara_slaes_project.gold.dimcustomer_info;"))
display(spark.sql("SELECT COUNT(*) FROM bara_slaes_project.gold.dimproduct_info;"))
display(spark.sql("SELECT COUNT(*) FROM bara_slaes_project.gold.factable;"))

# *Loading Data*

In [0]:
tables = {
    "sales": "bara_slaes_project.gold.factable",
    "Customer": "bara_slaes_project.gold.dimcustomer_info",
    "Product": "bara_slaes_project.gold.dimproduct_info",
}

cols_config = {
    "sales": ["Product_srgk","Customer_srgk","Order_Date", "Sales"],
    "Customer": ["Customer_srgk","Gender", "marital_status", "Country"],
    "Product": [
        "Product_srgk",
        "Product_ID",
        "Product_name",
        "Product_Line",
        "Category_Name",
        "Sub_Category_Name",
        "Maintenance",
    ],
}

In [0]:
def load_all_tables(tables, cols_config):

    data = {}

    for key, table in tables.items():
        df = spark.read.table(table)
        if key in cols_config:
            df = df.select(*cols_config[key])

        data[key] = df

    return data


data = load_all_tables(tables=tables, cols_config=cols_config)

In [0]:
def master_dataframe(data):

    df = data["sales"].join(data["Customer"],on="Customer_srgk").join(data["Product"],on="Product_srgk")
    df=df.drop("Product_srgk","Customer_srgk")

    return df


df = master_dataframe(data=data)

---

# *Data Engeneering*

In [0]:
from pyspark.sql.functions import col,year,monthname,month
df=(
    df
    .withColumn("order_year",year(col("Order_Date")))
    .withColumn("order_month",monthname(col("Order_Date")))
    .withColumn("order_month_number",month(col("Order_Date")))

)

In [0]:
df=df.select('Gender', 'marital_status', 'Country', 'Product_Line', 'Category_Name', 'Maintenance',"Sales")

In [0]:
df.count()

# *Convert spark dataframe to pandas*

In [0]:
pdf=df.toPandas()

In [0]:
import seaborn as sn
import matplotlib.pyplot as plt
import numpy as np
import pandas as pd
import seaborn as sns
import matplotlib.pyplot as plt
import plotly.express as px
import plotly.graph_objects as go

In [0]:
pdf.info()

*Detect numerical_cols,categorical_cols:*

In [0]:
numerical_cols=pdf.select_dtypes(include=np.number).columns.tolist()
categorical_cols=pdf.select_dtypes(include='object').columns.tolist()

print(f"numerical_cols : \n{numerical_cols}")
print("--------------------------------------------------")
print(f"categorical_cols : \n{categorical_cols}")
print("--------------------------------------------------")

---
*Exploratory Data Analysis-EDA :*  
---

* *Number of unique values in each feature :*

In [0]:
ax=pdf.nunique().to_frame().rename(columns={0:"Number of unique values"}).sort_values(by="Number of unique values",
                                                                                  ascending=False).plot(kind="barh",figsize=(20,7),title="Number of unique values in each feature");

ax.bar_label(ax.containers[0]);

* *numerical_cols :*

In [0]:
pdf[numerical_cols].describe().T


* *more numerical_cols basic summary statistics :*

In [0]:
from scipy import stats
confidence_level = 0.95
confidence_interval_list=[]

for i in numerical_cols:
    sample_mean = np.mean(pdf[i])
    standard_error_of_mean = stats.sem(pdf[i])
    degrees_of_freedom = len(pdf[i]) - 1
    confidence_interval = stats.t.interval(
        df=degrees_of_freedom,
        loc=sample_mean,
        scale=standard_error_of_mean,
        confidence=confidence_level
    )
        
    confidence_interval_list.append(confidence_interval)
    ci_df = pd.DataFrame(confidence_interval_list, columns=['CI_lower', 'CI_upper'])

    morestats=pd.concat([pdf[numerical_cols].var().to_frame().rename(columns={0:"variance"}),
            pdf[numerical_cols].std().to_frame().rename(columns={0:"std"}),
            pdf[numerical_cols].skew().to_frame().rename(columns={0:"skewness"}),
            pdf[numerical_cols].kurtosis().to_frame().rename(columns={0:"kurtosis"}),
            pdf[numerical_cols].sem().to_frame().rename(columns={0:"st error of the mean"})
            ],axis=1)

ci_df.index=numerical_cols
pd.concat([morestats,ci_df],axis=1)

* *Visalizing numerical_cols:*

In [0]:
for col in numerical_cols:
    plt.figure(figsize=(20, 4))

    plt.subplot(1, 3, 1)
    pdf[col].hist(bins=50)
    plt.title(f"{col} histogram plot")

    plt.subplot(1, 3, 2)
    sns.kdeplot(pdf[col], fill=True, color='steelblue')
    plt.title(f"{col} density function plot")

    plt.subplot(1, 3, 3)
    sns.ecdfplot(pdf[col], color='orange')
    plt.title(f"{col} cumulative density")

    plt.tight_layout()
    plt.show()

* *Categorical features :*

In [0]:
pdf[categorical_cols].describe().T

In [0]:
def plot_countplots_with_counts(df, columns,rotation_angle=45):
    """
    Generates countplots for multiple columns in a single figure with counts annotated on bars.

    Parameters:
    df (pd.DataFrame): The input DataFrame.
    columns (list): A list of column names to plot.
    """
    # Determine grid size (e.g., 2 rows and appropriate columns)
    if len(columns) != 6:
        print(f"Error: Expected 6 columns, but got {len(columns)}.")
        return

    # Set up the 3x3 subplot grid
    fig, axes = plt.subplots(nrows=2, ncols=3, figsize=(20, 12), tight_layout=True)
    
    # Flatten the 3x3 axes array into a 1D array for easy iteration
    axes_flat = axes.flatten()

    for i, col in enumerate(columns):
        ax = axes_flat[i]
        
        # Create the countplot on the specific axis
        sns.countplot(x=col, data=df, ax=ax)
        
        # Add counts to the bars using ax.bar_label()
        for container in ax.containers:
            ax.bar_label(container, fmt='%d') 
        
        # Rotate the x-axis labels
        ax.tick_params(axis='x', labelrotation=rotation_angle)
        
        ax.set_title(f'Countplot for {col}')
        ax.set_xlabel(col)
        ax.set_ylabel('Count')

    plt.show()

In [0]:
def plot_countplots_with_percent(df, columns, rotation_angle=45):
    """
    Generates annotated countplots for exactly 6 columns in a 2x3 grid 
    with percentages annotated on bars instead of counts.

    Parameters:
    df (pd.DataFrame): The input DataFrame.
    columns (list): A list of exactly 6 column names to plot.
    rotation_angle (int/float): The degree of rotation for x-axis labels.
    """
    if len(columns) != 6:
        print(f"Error: Expected 6 columns, but got {len(columns)}.")
        return

    fig, axes = plt.subplots(nrows=2, ncols=3, figsize=(20, 12), tight_layout=True)
    axes_flat = axes.flatten()

    for i, col in enumerate(columns):
        ax = axes_flat[i]
        
        # Calculate the total number of non-null observations for this column
        total = float(df[col].notna().sum())
        
        # Create the countplot
        sns.countplot(x=col, data=df, ax=ax)
        
        # Add percentage labels to the bars
        for container in ax.containers:
            labels = [f'{((h / total) * 100):.1f}%' for h in container.datavalues]
            
            # Removed 'labeltype="edge"' to avoid the AttributeError 
            # if using an older Matplotlib version
            ax.bar_label(container, labels=labels) 
        
        # Rotate the x-axis labels
        ax.tick_params(axis='x', labelrotation=rotation_angle)
        
        ax.set_title(f'Percentage Distribution of {col}')
        ax.set_xlabel(col)
        ax.set_ylabel('Count (Percentages Labeled)')

    plt.show()

In [0]:
plot_countplots_with_counts(df=pdf, columns=categorical_cols)

In [0]:
plot_countplots_with_percent(df=pdf, columns=categorical_cols, rotation_angle=45)

---
*Probabilities :*  
---

In [0]:
#col="Sales"
#hue_col = 'Gender' 

def distributionByCaregory(df,numberical_col,categorical_col):
    col=numberical_col
    hue_col = categorical_col

    plt.figure(figsize=(25, 5))

    plt.subplot(1, 3, 1)
    sns.histplot(data=df, x=col, hue=hue_col, bins=50, kde=False, multiple='stack',alpha=.7)
    plt.title(f"{col} histogram plot by {hue_col}")

    plt.subplot(1, 3, 2)
    sns.kdeplot(data=df, x=col, hue=hue_col, fill=True)
    plt.title(f"{col} density function plot by {hue_col}")

    plt.subplot(1, 3, 3)
    sns.ecdfplot(data=df, x=col, hue=hue_col)
    plt.title(f"{col} cumulative density  by {hue_col}")

    plt.tight_layout()
    plt.show()

In [0]:
for col in categorical_cols:
    distributionByCaregory(df=pdf,numberical_col="Sales",categorical_col=col)

In [0]:
%run /Workspace/Users/omars.soub@gmail.com/DataBricks_Baraa_sales_Project/Analysis/Detailed_Python_Analysis/utils

In [0]:
price_conditional_empirical_prob_all(df=pdf, price_col="Sales", threshold=500)


In [0]:
plot_price_distributions(df=pdf, price_col="Sales", threshold=500)

---
*Inferential statistics :*  
--

* *Hypothesis testing: Checking the Target feature-Price- normality*

In [0]:
from scipy.stats import shapiro
stat, p = shapiro(pdf["Sales"])
print(f"W = {stat:.3f}, p = {p:.3f}")
if p < 0.05:
    print("Reject H₀ → distribution is not normal")
else:
    print("Fail to reject H₀ → distribution is compatible with normality")

fig, axes = plt.subplots(1, 3, figsize=(20, 5))   
sns.boxplot(data=pdf, x="Sales", ax=axes[0])
axes[0].set_title('Sales Boxplot')  
sns.histplot(data=pdf, x="Sales", ax=axes[1])
axes[1].set_title(f'Sales histogram') 
stats.probplot(pdf["Sales"], dist="norm", plot=plt)
axes[2].set_title(f'Sales qq-plot')    

*As the Target feature-Sales is not normaly distributed , we will apply non-parametric statistical tests*  
*--> Null hypothesis :There is no significant relationship / distribution difference between the P-value shoud be > .05*  
*--> Alternatice hypothesis :There is significant relationship / distribution difference between the P-value shoud be < .05*

In [0]:
def kruskal_func(Dataset:pd.DataFrame,Numericaltarget:str,CategoricalFeature:str):
    from scipy.stats import kruskal
    groups = [Dataset[Dataset[CategoricalFeature] == level][Numericaltarget] for level in Dataset[CategoricalFeature].unique()]

    stat, p = kruskal(*groups)
    if p < 0.05: 
        print(f"Kruskal-Wallis Test: Statistic={stat:.3f}, P-value={p:.3f} ==>\n The p value less than 0.05 ==> At least one group has Significant difference")
        fig, axes = plt.subplots(1, 2, figsize=(20, 5))
        sns.boxplot(x=CategoricalFeature, y=Numericaltarget, data=Dataset, palette="Set2",ax=axes[0])
        sns.stripplot(x=CategoricalFeature, y=Numericaltarget, data=Dataset, color="black", alpha=0.6, jitter=True,ax=axes[1])
        plt.title(f"Kruskal–Wallis Test\nH={stat:.2f}, p={p:.4f}")
        

        print("As At least one group has Significant difference --> posthoc_dunn test")
        posthoc_dunn_test(Dataset,Numericaltarget,CategoricalFeature)
        plt.show()

    else:
        print(f"Kruskal-Wallis Test: Statistic={stat:.3f}, P-value={p:.3f} ==>\n the p value more than 0.05 ==> No Significant difference between groups")

In [0]:
for col in categorical_cols:
    print(f"Sales VS {col}")
    kruskal_func(Dataset=pdf,Numericaltarget="Sales",CategoricalFeature=col)
    print("")

===================================================================